# Djinni Step 4 Final
Notebook chốt final từ file Step 3 sạch (`07_djinni_step3_review_fixed_v2.xlsx`).

In [1]:
from IPython.display import display


In [4]:

# ==== STEP 4: CHỐT FINAL DJINNI ====
import pandas as pd
import numpy as np

INPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/07_djinni_step3_review_fixed_v2.xlsx"
OUTPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/08_djinni_step4_final.xlsx"

# ==== ĐỌC FILE ====
df = pd.read_excel(INPUT_FILE, sheet_name="Sheet1").copy()

# ==== CHUẨN HOÁ ====
for col in [
    "ten_ngoai_thi_truong",
    "cac_ten_gan_giong",
    "huong_xu_ly",
    "ghi_chu_djinni",
]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str).str.strip()

if "row_id" not in df.columns:
    df["row_id"] = range(2, len(df) + 2)

# ==== QUYẾT ĐỊNH STEP 4: GỠ CÁC CẶP TÊN THỊ TRƯỜNG BỊ TRÙNG ====
# row_id -> (ten_moi, ly_do)
step4_decisions = {
    616: ("cloud security compliance implementation", "Tách khỏi base concept 'cloud security'."),
    834: ("online data analysis", "Tách khỏi 'data analysis' chung."),
    1105: ("information security strategy development", "Tách khỏi concept danh từ."),
    113: ("security legislation", "Tách khỏi 'security compliance'."),
    819: ("security systems planning", "Tách khỏi 'security planning' chung."),
    1104: ("information security policy implementation", "Tách khỏi policy implementation chung."),
    685: ("software test levels", "Tách khỏi hoạt động 'software testing'."),
    453: ("test suite development", "Tách khỏi 'test automation'."),
}

change_logs = []
for row_id, (new_name, reason) in step4_decisions.items():
    mask = df["row_id"].eq(row_id)
    if mask.any():
        old_name = df.loc[mask, "ten_ngoai_thi_truong"].iloc[0]
        df.loc[mask, "ten_ngoai_thi_truong"] = new_name
        old_note = df.loc[mask, "ghi_chu_djinni"].iloc[0]
        note_parts = [x for x in [old_note, f"step4_dedup: {old_name} -> {new_name}"] if str(x).strip()]
        df.loc[mask, "ghi_chu_djinni"] = " | ".join(note_parts)
        change_logs.append([row_id, old_name, new_name, reason])

# ==== TÍNH LẠI CỜ REVIEW ====
df["flag_missing_market_name"] = (
    (df["huong_xu_ly"] == "doi_ten") & (df["ten_ngoai_thi_truong"] == "")
).astype(int)

df["flag_missing_alias"] = (
    (df["huong_xu_ly"] == "them_ten_gan_giong") & (df["cac_ten_gan_giong"] == "")
).astype(int)

df["market_name_norm"] = df["ten_ngoai_thi_truong"].str.strip().str.lower()
market_name_counts = df.loc[df["market_name_norm"] != "", "market_name_norm"].value_counts()
duplicate_market_names = set(market_name_counts[market_name_counts > 1].index)

df["flag_duplicate_market_name"] = df["market_name_norm"].isin(duplicate_market_names).astype(int)

df["review_issue"] = ""
mask = df["flag_missing_market_name"].eq(1)
df.loc[mask, "review_issue"] = "thieu_ten_ngoai_thi_truong"

mask = df["flag_missing_alias"].eq(1)
df.loc[mask, "review_issue"] = np.where(
    df.loc[mask, "review_issue"].eq(""),
    "thieu_cac_ten_gan_giong",
    df.loc[mask, "review_issue"] + "; thieu_cac_ten_gan_giong",
)

mask = df["flag_duplicate_market_name"].eq(1)
df.loc[mask, "review_issue"] = np.where(
    df.loc[mask, "review_issue"].eq(""),
    "trung_ten_ngoai_thi_truong",
    df.loc[mask, "review_issue"] + "; trung_ten_ngoai_thi_truong",
)

df["can_xem_thu_cong"] = df["huong_xu_ly"].isin(["doi_ten", "them_ten_gan_giong"]).astype(int)
df["review_priority"] = np.where(
    df["review_issue"].ne(""),
    "issue",
    np.where(df["can_xem_thu_cong"].eq(1), "can_xem", ""),
)

# ==== CẬP NHẬT TRẠNG THÁI FINAL ====
df["da_kiem"] = 1
df["vong"] = 2
df["buoc"] = 4

# ==== TẠO SHEET REVIEW STEP 4 ====
review_df = df[
    (df["can_xem_thu_cong"].eq(1)) | (df["review_issue"].ne(""))
].copy()

priority_order = {"issue": 0, "can_xem": 1}
review_df["priority_rank"] = review_df["review_priority"].map(priority_order).fillna(99)

review_df = (
    review_df.sort_values(by=["priority_rank", "row_id"], ascending=[True, True])
    .drop(columns=["priority_rank"])
    .reset_index(drop=True)
)

# ==== SUMMARY ====
summary = {
    "tong_so_dong": int(len(df)),
    "giu_nguyen": int((df["huong_xu_ly"] == "giu_nguyen").sum()),
    "doi_ten": int((df["huong_xu_ly"] == "doi_ten").sum()),
    "them_ten_gan_giong": int((df["huong_xu_ly"] == "them_ten_gan_giong").sum()),
    "dong_can_xem_thu_cong": int(df["can_xem_thu_cong"].sum()),
    "dong_co_issue": int((df["review_issue"] != "").sum()),
    "issue_thieu_ten_ngoai_thi_truong": int(df["flag_missing_market_name"].sum()),
    "issue_thieu_cac_ten_gan_giong": int(df["flag_missing_alias"].sum()),
    "issue_trung_ten_ngoai_thi_truong": int(df["flag_duplicate_market_name"].sum()),
    "so_ten_ngoai_thi_truong_bi_trung_unique": int(len(duplicate_market_names)),
    "so_dong_duoc_sua_step4": int(len(change_logs)),
}

summary_df = pd.DataFrame(list(summary.items()), columns=["chi_so", "gia_tri"])
changes_df = pd.DataFrame(change_logs, columns=["row_id", "ten_thi_truong_cu", "ten_thi_truong_moi", "ly_do"])

# ==== GHI FILE EXCEL ====
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Sheet1")
    review_df.to_excel(writer, index=False, sheet_name="review_buoc_4")
    summary_df.to_excel(writer, index=False, sheet_name="summary_buoc_4")
    changes_df.to_excel(writer, index=False, sheet_name="step4_changes")

print(f"Đã tạo file: {OUTPUT_FILE}")

print("\n=== SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n=== STEP 4 CHANGES ===")
display(changes_df)

print("\n=== TOP REVIEW ROWS ===")
display(
    review_df[
        [
            "row_id",
            "huong_xu_ly",
            "ten_goc",
            "ten_ngoai_thi_truong",
            "cac_ten_gan_giong",
            "review_issue",
            "review_priority",
        ]
    ].head(20).fillna("")
)


Đã tạo file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/08_djinni_step4_final.xlsx

=== SUMMARY ===
tong_so_dong: 1171
giu_nguyen: 1116
doi_ten: 46
them_ten_gan_giong: 9
dong_can_xem_thu_cong: 55
dong_co_issue: 0
issue_thieu_ten_ngoai_thi_truong: 0
issue_thieu_cac_ten_gan_giong: 0
issue_trung_ten_ngoai_thi_truong: 0
so_ten_ngoai_thi_truong_bi_trung_unique: 0
so_dong_duoc_sua_step4: 8

=== STEP 4 CHANGES ===


,row_id,ten_thi_truong_cu,ten_thi_truong_moi,ly_do
0,616,cloud security,cloud security compliance implementation,Tách khỏi base concept 'cloud security'.
1,834,data analysis,online data analysis,Tách khỏi 'data analysis' chung.
2,1105,information security strategy,information security strategy development,Tách khỏi concept danh từ.
3,113,security compliance,security legislation,Tách khỏi 'security compliance'.
4,819,security planning,security systems planning,Tách khỏi 'security planning' chung.
5,1104,security policy implementation,information security policy implementation,Tách khỏi policy implementation chung.
6,685,software testing,software test levels,Tách khỏi hoạt động 'software testing'.
7,453,test automation,test suite development,Tách khỏi 'test automation'.



=== TOP REVIEW ROWS ===


,row_id,huong_xu_ly,ten_goc,ten_ngoai_thi_truong,cac_ten_gan_giong,review_issue,review_priority
0,5,them_ten_gan_giong,machine learning,machine learning,"ml, ai, model training",,can_xem
1,77,them_ten_gan_giong,DevOps,devops,"sre, ci/cd, platform engineering",,can_xem
2,103,doi_ten,ICT network security risks,network security,"network security risks, network risk, cyber risk",,can_xem
3,113,doi_ten,ICT security legislation,security legislation,"security legislation, it security law, cyberse...",,can_xem
4,114,doi_ten,ICT security standards,security standards,"cybersecurity standards, information security ...",,can_xem
5,197,doi_ten,advice on security risk management,security risk management,"cyber risk management, security risk advisory,...",,can_xem
6,278,doi_ten,blockchain applications security principles,blockchain security,"blockchain application security, web3 security...",,can_xem
7,319,doi_ten,cloud security and compliance,cloud security,"cloud compliance, cloud security and complianc...",,can_xem
8,349,doi_ten,conduct textile testing operations,textile testing,"textile testing operations, textile quality te...",,can_xem
9,395,them_ten_gan_giong,cyber security,cybersecurity,"cyber security, information security, infosec",,can_xem
